<a href="https://colab.research.google.com/github/kongyoubuaa/statistics_homework_2026_BUCT/blob/master/%E7%BB%9F%E8%AE%A1%E5%AD%A6%E4%BD%9C%E4%B8%9A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Exercise 1
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

file_path = 'hw2ex1.csv'
df = pd.read_csv(file_path)

# Clean up column names by stripping whitespace and converting to lowercase
df.columns = df.columns.str.strip().str.lower()
df = df.apply(pd.to_numeric, errors='ignore')

# 结构化整理数据
total_num = df.loc[df['体重'] == '合计', '职工人数'].iloc[0]
df.drop(df.tail(1).index, inplace = True)
split_cols = df['体重'].str.split('-', expand = True)
split_cols.columns = ['start', 'end']
try:
  split_cols = split_cols.apply(pd.to_numeric)
except:
  pass

df = pd.concat([df, split_cols], axis=1)
df.drop('体重', axis = 1, inplace = True)

# 调试程序
# print(total_num)
# print(df)

# (1).职工体重的平均数
total_weight = 0
for row in df.itertuples():
  average_line = (row.start + row.end) / 2
  total_weight += average_line * row.职工人数
print('职工体重的平均数为：{}'.format(total_weight / total_num))

# (2).职工体重的中位数
median_pos = total_num / 2
# median_pos_lower 为低于中位数所属区间的人数
# median_pos_upper 为高于中位数所属区间的人数
# median_pos_index 为中位数所属区间的行号
median_pos_lower, median_pos_upper, median_pos_index = 0, 0, 0
index_tmp = 0
for i in df['职工人数'].values[::1]:
  if median_pos_lower + i >= median_pos:
    median_pos_index = index_tmp
    break;
  else:
    median_pos_lower += i
    index_tmp += 1
index_tmp = 0
for i in df['职工人数'].values[::-1]:
  if index_tmp == median_pos_index:
    break;
  else:
    median_pos_upper += i
    index_tmp += 1
# 计算上下界
# 因为体重在区间内连续分布，概率相同（均匀分布, f'=0），故不需要考虑分布导致的偏移
median_range, median_range_start, median_range_end = \
  df.at[median_pos_index, '职工人数'], df.at[median_pos_index, 'start'], \
  df.at[median_pos_index, 'end']
# ！！！ 下界的计算 ！！！
median_lower = median_range_start + (median_pos - median_pos_lower) \
  / median_range * (median_range_end - median_range_start)
# ！！！ 上界的计算 ！！！
median_upper = median_range_end - (median_pos - median_pos_upper) \
  / median_range * (median_range_end - median_range_start)
# 浮点数比大小考虑误差
if abs(median_upper - median_lower) < 1e-9:
  print('职工体重的中位数为：{:.2f}'.format(median_upper))
else:
  print('职工体重的中位数上界为：{}，下界为：{}'.format(median_upper, median_lower))

# (3).职工体重的众数
modal_class_index = df['职工人数'].idxmax()
modal_pos_start = df.at[modal_class_index, 'start'] # 区间下界
modal_pos_end = df.at[modal_class_index, 'end'] # 区间上界
modal_pos_length = modal_pos_end - modal_pos_start # 组距
modal_pos_num = df.at[modal_class_index, '职工人数']
try: # 众数左相邻区间频数
  modal_left_num = df.at[modal_class_index - 1, '职工人数']
except: # 如无左区间，则记为0
  modal_left_num = 0
try: # 众数右相邻区间频数
  modal_right_num = df.at[modal_class_index + 1, '职工人数']
except: # 如无右区间，则记为0
  modal_right_num = 0

# 众数计算（从下界）
modal_lower = modal_pos_start + (modal_pos_num - modal_left_num) \
  / ((modal_pos_num - modal_left_num) + (modal_pos_num - modal_right_num)) \
  * modal_pos_length

# 众数计算（从上界）
modal_upper = modal_pos_end - (modal_pos_num - modal_right_num) \
  / ((modal_pos_num - modal_left_num) + (modal_pos_num - modal_right_num)) \
  * modal_pos_length

# 浮点数比大小考虑误差
if abs(modal_upper - modal_lower) < 1e-9:
  print('职工体重的众数为：{:.2f}'.format(modal_upper))
else:
  print('职工体重的众数上界为：{}，下界为：{}'.format(modal_upper, modal_lower))


# 过程变量集中输出：
print("以下为过程中各变量值，具体计算请参考后附Github中代码源文件")
print("体重总重量：{}".format(total_weight))
print("合计总人数：{}".format(total_num))
print("体重中位人数位置：{}".format(median_pos))
print("低于中位数所属区间的人数：{}".format(median_pos_lower))
print("高于中位数所属区间的人数：{}".format(median_pos_upper))
print("中位数所属区间的组号位置：{}".format(median_pos_index + 1))
print("体重众数人数位置：{}".format(modal_class_index + 1))
print("体重众数组内人数：{}".format(modal_pos_num))
print("体重众数左侧组内人数：{}".format(modal_left_num))
print("体重众数右侧组内人数：{}".format(modal_right_num))


职工体重的平均数为：75.6
职工体重的中位数为：75.71
职工体重的众数为：76.25
以下为过程中各变量值，具体计算请参考后附Github中代码源文件
体重总重量：7560.0
合计总人数：100
体重中位人数位置：50.0
低于中位数所属区间的人数：34
高于中位数所属区间的人数：38
中位数所属区间的组号位置：4
体重众数人数位置：4
体重众数组内人数：28
体重众数左侧组内人数：18
体重众数右侧组内人数：22


In [31]:
# Exercise 2
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import re

file_path = 'hw2ex2.csv'
df = pd.read_csv(file_path, header=None)
df = list(df.itertuples(index=False, name=None))

# 输出结构化数据备查
print('数据预处理结果：{}'.format(df))

# (1). 计算各组的标准差
a_list = df[0][1:]
b_list = df[1][1:]
# 两组数据均值
a_average = sum(a_list) / (len(a_list))
b_average = sum(b_list) / (len(b_list))
# 甲组标准差计算
a_var_quantity = 0 # 计算方差且不除数量
for i in a_list:
  a_var_quantity += pow(i - a_average, 2)
a_std = pow(a_var_quantity / len(a_list), 1 / 2)
print('甲组标准差为：{:.2f}'.format(a_std))
# 乙组标准差
b_var_quantity = 0 # 计算方差且不除数量
for i in b_list:
  b_var_quantity += pow(i - b_average, 2)
b_std = pow(b_var_quantity / len(b_list), 1 / 2)
print('乙组标准差为：{:.2f}'.format(b_std))


# (2). 比较两组工人平均日产量稳定性。
a_stability = a_std / a_average
b_stability = b_std / b_average
print('甲组稳定性为：{:.2f}%，乙组稳定性为：{:.2f}%'.format(a_stability * 100, b_stability * 100))
print('因此：', end='')
if abs(a_stability - b_stability) < 1e-9:
  print('两组工人平均日产量稳定性相同')
elif a_stability < b_stability:
  print('甲组工人平均日产量稳定性高于乙组工人')
else :
  print('乙组工人平均日产量稳定性高于甲组工人')

数据预处理结果：[('甲', 20, 40, 60, 80, 100, 120), ('乙', 67, 68, 69, 71, 72, 73)]
甲组标准差为：34.16
乙组标准差为：2.16
甲组稳定性为：48.80%，乙组稳定性为：3.09%
因此：乙组工人平均日产量稳定性高于甲组工人
